In [1]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

from utils import protein
from utils.geometry import compute_rmsd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [2]:
motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
root_dir = f"./out/onemotif_twostates/{motif}/"

rows = []
for design_dir in sorted(glob.glob(os.path.join(root_dir, "design*"))):
    design_name = os.path.basename(design_dir)
    
    with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
        motif_mask = pickle.load(f)["motif_mask"]

    for state in [0, 1]:
        for sample_idx in range(5): 
            pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
            if not os.path.exists(pdb_file):
                continue
            rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)
            rows.append({
                "design": design_name,
                "state": state,
                "sample": sample_idx,
                "rmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
            })

df = pd.DataFrame(rows)
df


,design,state,sample,rmsd
0,design0,0,0,0.824371
1,design0,0,1,0.888581
2,design0,0,2,0.853081
3,design0,0,3,0.945256
4,design0,0,4,0.810755
5,design0,1,0,1.479519
6,design0,1,1,1.487967
7,design0,1,2,1.589290
8,design0,1,3,1.484051
9,design0,1,4,1.573743


In [5]:
mean_rmsds = df.groupby(["design", "state"])["rmsd"].mean().reset_index()
mean_rmsds = mean_rmsds.pivot(index="design", columns="state", values="rmsd").reset_index()
mean_rmsds.columns.name = None 
mean_rmsds = mean_rmsds.rename(columns={0: "unbound state", 1: "bound state"})
mean_rmsds

,design,unbound state,bound state
0,design0,0.864409,1.522914
1,design1,0.463217,4.329279
2,design2,0.509397,1.382533
3,design3,2.413548,2.309627


In [6]:
fig = px.scatter(
    mean_rmsds,
    x="unbound state",
    y="bound state",
    title="Mean motif RMSD Å"
)
fig.update_traces(textposition="top center")
fig.add_shape(type="line", x0=0, y0=0, x1=5, y1=5,
              line=dict(color="black", dash="dash"))
fig.update_layout(
    xaxis=dict(range=[0, 5]),
    yaxis=dict(range=[0, 5], scaleanchor="x"),
    width=600, height=600
)
